# Geometric Bioacoustics: Hyperbolic Species Classification

**Core idea**: Species are taxonomically hierarchical (order > family > genus > species). Hyperbolic geometry naturally encodes trees with exponentially less distortion than Euclidean space. We exploit this by:

1. **Poincaré ball classification head** — replaces the standard softmax layer with hyperbolic multinomial logistic regression, where class prototypes are initialized from the taxonomic tree
2. **SPD manifold features** — frequency-band covariance matrices live on a Riemannian manifold; the log-Euclidean metric captures second-order spectral statistics that CNNs miss
3. **TDA analysis** — persistent homology on time-delay embeddings captures the topology of the vocal apparatus attractor

The backbone is EfficientNet on mel-spectrograms (proven in BirdCLEF competitions). The geometric components target **rare species** where macro-averaged AUC is won or lost.

---

In [ ]:
!pip install timm torchaudio librosa ripser persim scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import librosa
import timm
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
@dataclass
class Config:
    # Audio
    sr: int = 32000
    duration: float = 5.0
    n_mels: int = 128
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 50
    fmax: int = 14000

    # Model
    backbone: str = "tf_efficientnetv2_s"  # standard BirdCLEF backbone
    embed_dim: int = 128
    hyperbolic_c: float = 1.0  # Poincare ball curvature

    # Training
    batch_size: int = 32
    lr: float = 1e-3
    epochs: int = 30

    # SPD manifold
    spd_n_bands: int = 16  # frequency band groups for covariance

cfg = Config()
print(f"Spectrogram shape: [{cfg.n_mels}, {int(cfg.duration * cfg.sr / cfg.hop_length)}]")

## Part 1: Poincaré Ball Geometry

The Poincaré ball $\mathbb{B}^d_c = \{x \in \mathbb{R}^d : c\|x\|^2 < 1\}$ is a model of hyperbolic space where:
- Points near the **center** have low curvature (broad categories like "order")
- Points near the **boundary** have high curvature (fine-grained species)
- A tree with $n$ leaves can be embedded with $O(\log n)$ distortion vs $O(n)$ in Euclidean space

This makes it ideal for taxonomic classification where the number of species per genus varies wildly.

In [ ]:
class PoincareBall:
    """Poincare ball model of hyperbolic space.

    Points live inside the ball {x : c||x||^2 < 1}.
    Operations use the Mobius gyrovector formalism.
    """

    def __init__(self, c=1.0):
        self.c = c
        self.sqrt_c = c ** 0.5

    def project(self, x, eps=1e-5):
        """Project onto the open ball (clamp norm < 1/sqrt(c))."""
        max_norm = (1.0 / self.sqrt_c) - eps
        norm = x.norm(dim=-1, keepdim=True)
        return torch.where(norm > max_norm, x / norm * max_norm, x)

    def expmap0(self, v):
        """Exponential map from origin: tangent vector -> ball."""
        v_norm = v.norm(dim=-1, keepdim=True).clamp_min(1e-15)
        return self.project(
            torch.tanh(self.sqrt_c * v_norm) * v / (self.sqrt_c * v_norm)
        )

    def logmap0(self, y):
        """Logarithmic map to origin: ball -> tangent space."""
        y_norm = y.norm(dim=-1, keepdim=True).clamp(1e-15, (1 / self.sqrt_c) - 1e-5)
        return torch.arctanh(self.sqrt_c * y_norm) * y / (self.sqrt_c * y_norm)

    def mobius_add(self, x, y):
        """Mobius addition: the hyperbolic analogue of vector addition."""
        x2 = (x * x).sum(-1, keepdim=True)
        y2 = (y * y).sum(-1, keepdim=True)
        xy = (x * y).sum(-1, keepdim=True)
        num = (1 + 2 * self.c * xy + self.c * y2) * x + (1 - self.c * x2) * y
        denom = 1 + 2 * self.c * xy + self.c ** 2 * x2 * y2
        return self.project(num / denom.clamp_min(1e-15))

    def dist(self, x, y):
        """Hyperbolic distance via Mobius operations."""
        # d(x,y) = (2/sqrt(c)) * arctanh(sqrt(c) * ||-x oplus y||)
        neg_x = -x
        add_result = self.mobius_add(neg_x, y)
        norm = add_result.norm(dim=-1).clamp(max=(1 / self.sqrt_c) - 1e-5)
        return (2.0 / self.sqrt_c) * torch.arctanh(self.sqrt_c * norm)


class HyperbolicMLR(nn.Module):
    """Hyperbolic Multinomial Logistic Regression.

    Classification via distance to learned prototypes on the Poincare ball.
    Prototypes can be initialized from taxonomic embeddings so that
    species in the same genus start close together.
    """

    def __init__(self, embed_dim: int, num_classes: int, c: float = 1.0):
        super().__init__()
        self.ball = PoincareBall(c)
        # Prototypes live in tangent space; mapped to ball via expmap
        self.proto_tangent = nn.Parameter(torch.randn(num_classes, embed_dim) * 0.01)
        # Per-class learnable temperature (helps rare species)
        self.log_scale = nn.Parameter(torch.zeros(num_classes))

    @property
    def prototypes(self):
        return self.ball.expmap0(self.proto_tangent)

    def forward(self, x):
        """x: [B, D] points on the Poincare ball -> logits [B, C]."""
        protos = self.prototypes  # [C, D]
        # Batch distance: [B, 1, D] vs [1, C, D]
        dists = self.ball.dist(x.unsqueeze(1), protos.unsqueeze(0))  # [B, C]
        return -dists * self.log_scale.exp()

    def init_from_taxonomy(self, embeddings: torch.Tensor):
        """Initialize prototypes from taxonomic tree embeddings."""
        with torch.no_grad():
            # Scale embeddings into the ball, then invert to tangent space
            on_ball = self.ball.project(embeddings * 0.5)
            self.proto_tangent.copy_(self.ball.logmap0(on_ball))


# Quick sanity check
ball = PoincareBall(c=1.0)
x = ball.expmap0(torch.randn(4, 8) * 0.3)
y = ball.expmap0(torch.randn(4, 8) * 0.3)
d = ball.dist(x, y)
print(f"Poincare ball test — norms: {x.norm(dim=-1).tolist()}")
print(f"Distances: {d.tolist()}")
print(f"All inside ball: {(x.norm(dim=-1) < 1).all().item()}")

## Part 2: Taxonomy Embedding

We create a synthetic 8-species taxonomy and embed it in hyperbolic space.
Species in the same family are placed close together on the Poincaré ball;
different orders are far apart. These embeddings initialize the classification
prototypes — giving the model a **structural prior** before seeing any audio.

In [ ]:
def build_synthetic_taxonomy():
    """8 synthetic species organized into orders and families."""
    taxonomy = {
        'Passeriformes': {
            'Parulidae': ['spectral_warbler'],
            'Fringillidae': ['pulse_finch', 'trill_sparrow'],
            'Turdidae': ['harmonic_thrush'],
            'Troglodytidae': ['chirp_wren'],
            'Tyrannidae': ['buzz_flycatcher'],
        },
        'Psittaciformes': {
            'Psittacidae': ['click_parrot'],
        },
        'Apodiformes': {
            'Thraupidae': ['whistle_tanager'],
        },
    }

    species_list = []
    species_info = {}
    for order, families in taxonomy.items():
        for family, species in families.items():
            for sp in species:
                species_list.append(sp)
                species_info[sp] = {'idx': len(species_list)-1, 'order': order, 'family': family}

    n = len(species_list)
    dist_matrix = np.zeros((n, n))
    for i, si in enumerate(species_list):
        for j, sj in enumerate(species_list):
            if i == j:
                dist_matrix[i, j] = 0
            elif species_info[si]['family'] == species_info[sj]['family']:
                dist_matrix[i, j] = 1
            elif species_info[si]['order'] == species_info[sj]['order']:
                dist_matrix[i, j] = 2
            else:
                dist_matrix[i, j] = 3

    return species_list, species_info, taxonomy, dist_matrix


def embed_taxonomy_hyperbolic(dist_matrix: np.ndarray, embed_dim: int, c: float = 1.0):
    """Embed taxonomic distances into the Poincare ball via spectral kernel method.

    For a real competition, use Sarkar's algorithm or learned embeddings.
    This simplified version preserves relative distances well enough.
    """
    K = np.exp(-dist_matrix ** 2 / 2.0)  # Gaussian kernel
    eigvals, eigvecs = np.linalg.eigh(K)
    idx = np.argsort(eigvals)[::-1][:embed_dim]
    coords = eigvecs[:, idx] * np.sqrt(np.abs(eigvals[idx]))
    # Scale to fit inside ball with margin
    max_norm = np.max(np.linalg.norm(coords, axis=1))
    if max_norm > 0:
        coords = coords / max_norm * 0.7 / (c ** 0.5)
    return torch.tensor(coords, dtype=torch.float32)


# Build and visualize
species_list, species_info, taxonomy, dist_matrix = build_synthetic_taxonomy()
num_classes = len(species_list)
tax_embeddings = embed_taxonomy_hyperbolic(dist_matrix, cfg.embed_dim, cfg.hyperbolic_c)

print(f"Species ({num_classes}):")
for sp in species_list:
    info = species_info[sp]
    print(f"  {sp:20s}  {info['order']:20s}  {info['family']}")
print(f"\nTaxonomy embedding shape: {tax_embeddings.shape}")
print(f"Embedding norms: {tax_embeddings.norm(dim=-1).tolist()}")

## Part 3: Synthetic Audio Dataset

Each species gets a distinct vocalization pattern generated from first principles:
frequency sweeps, FM trills, harmonic stacks, pulse trains, clicks, and buzzes.
Random jitter ensures each sample is unique. Replace this with real BirdCLEF data
for competition use (see the integration guide at the end).

In [ ]:
def generate_bird_call(species: str, duration: float, sr: int) -> np.ndarray:
    """Generate a synthetic bird vocalization with species-specific acoustic signature."""
    n_samples = int(duration * sr)
    t = np.linspace(0, duration, n_samples)
    signal = np.zeros(n_samples)
    rng = np.random
    jitter = lambda x, pct=0.15: x * (1 + rng.uniform(-pct, pct))

    if species == 'spectral_warbler':
        # Ascending frequency sweep with harmonics
        f0, f1 = jitter(2000), jitter(6000)
        freq = np.linspace(f0, f1, n_samples)
        phase = 2 * np.pi * np.cumsum(freq) / sr
        signal = 0.7 * np.sin(phase) + 0.3 * np.sin(2 * phase)
        signal *= np.sin(np.pi * t / duration) ** 2

    elif species == 'pulse_finch':
        # Rapid repeated pulses
        rate, width, f0 = jitter(15), jitter(0.015), jitter(4500)
        for pt in np.arange(0, duration, 1.0 / rate):
            signal += np.exp(-((t - pt) / width) ** 2) * np.sin(2 * np.pi * f0 * t)

    elif species == 'harmonic_thrush':
        # Rich harmonics with vibrato
        f0 = jitter(1200)
        vibrato = 5 + 3 * np.sin(2 * np.pi * 6 * t)
        phase = 2 * np.pi * np.cumsum(f0 + vibrato) / sr
        for h in range(1, 6):
            signal += (1.0 / h) * np.sin(h * phase)
        env = np.ones(n_samples)
        env[:int(0.05 * sr)] = np.linspace(0, 1, int(0.05 * sr))
        env[-int(0.1 * sr):] = np.linspace(1, 0, int(0.1 * sr))
        signal *= env

    elif species == 'trill_sparrow':
        # FM trill
        fc, rate, depth = jitter(3500), jitter(25), jitter(800)
        freq = fc + depth * np.sin(2 * np.pi * rate * t)
        signal = np.sin(2 * np.pi * np.cumsum(freq) / sr)
        signal *= 0.5 + 0.5 * np.sin(2 * np.pi * rate * t)

    elif species == 'chirp_wren':
        # Short chirps at varying pitches
        for ct in np.sort(rng.uniform(0.2, duration - 0.2, int(jitter(8)))):
            f, w = jitter(5000), jitter(0.02)
            mask = np.exp(-((t - ct) / w) ** 2)
            signal += mask * np.sin(2 * np.pi * (f + 2000 * (t - ct) / w) * t)

    elif species == 'buzz_flycatcher':
        # Broadband buzz in two bursts
        f0 = jitter(2500)
        raw = np.sin(2 * np.pi * f0 * t) + 0.3 * np.sign(np.sin(2 * np.pi * jitter(100) * t))
        env = np.zeros(n_samples)
        mid = n_samples // 2
        bl = int(0.3 * sr)
        env[mid // 2 - bl // 2:mid // 2 + bl // 2] = 1.0
        env[mid + mid // 2 - bl // 2:mid + mid // 2 + bl // 2] = 1.0
        signal = raw * env

    elif species == 'click_parrot':
        # Sharp clicks with damped resonance
        f0 = jitter(3000)
        for ct in np.sort(rng.uniform(0.1, duration - 0.1, int(jitter(12)))):
            mask = np.exp(-50 * np.maximum(t - ct, 0)) * (t >= ct).astype(float)
            signal += mask * np.sin(2 * np.pi * f0 * (t - ct))

    elif species == 'whistle_tanager':
        # Pure tone with slight warble
        f0 = jitter(4000)
        phase = 2 * np.pi * np.cumsum(f0 + 50 * np.sin(2 * np.pi * 3 * t)) / sr
        signal = np.sin(phase) * np.sin(np.pi * t / duration)

    # Normalize and add noise
    peak = np.abs(signal).max()
    if peak > 0:
        signal = signal / peak * 0.8
    signal += rng.uniform(0.02, 0.08) * rng.randn(n_samples)

    return signal.astype(np.float32)


class SyntheticBirdDataset(Dataset):
    """Generates synthetic bird calls on the fly."""

    def __init__(self, species_list, samples_per_species, cfg, augment=False):
        self.species_list = species_list
        self.sps = samples_per_species
        self.cfg = cfg
        self.augment = augment

    def __len__(self):
        return len(self.species_list) * self.sps

    def __getitem__(self, idx):
        species_idx = idx // self.sps
        species = self.species_list[species_idx]

        audio = generate_bird_call(species, self.cfg.duration, self.cfg.sr)

        mel = librosa.feature.melspectrogram(
            y=audio, sr=self.cfg.sr, n_mels=self.cfg.n_mels,
            n_fft=self.cfg.n_fft, hop_length=self.cfg.hop_length,
            fmin=self.cfg.fmin, fmax=self.cfg.fmax
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

        if self.augment:
            # Time masking
            if np.random.random() < 0.5:
                ts = np.random.randint(0, max(1, mel_db.shape[1] - 20))
                mel_db[:, ts:ts + np.random.randint(5, 20)] = 0
            # Frequency masking
            if np.random.random() < 0.5:
                fs = np.random.randint(0, max(1, mel_db.shape[0] - 15))
                mel_db[fs:fs + np.random.randint(5, 15), :] = 0

        # 3-channel for EfficientNet
        mel_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0).expand(3, -1, -1)
        return mel_tensor, species_idx


# Preview: generate one call per species and plot spectrograms
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, (sp, ax) in enumerate(zip(species_list, axes.flat)):
    audio = generate_bird_call(sp, cfg.duration, cfg.sr)
    mel = librosa.feature.melspectrogram(
        y=audio, sr=cfg.sr, n_mels=cfg.n_mels,
        n_fft=cfg.n_fft, hop_length=cfg.hop_length,
        fmin=cfg.fmin, fmax=cfg.fmax
    )
    librosa.display.specshow(
        librosa.power_to_db(mel, ref=np.max),
        sr=cfg.sr, hop_length=cfg.hop_length,
        x_axis='time', y_axis='mel', ax=ax, fmin=cfg.fmin, fmax=cfg.fmax
    )
    ax.set_title(sp.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.suptitle('Synthetic Bird Vocalizations — Mel Spectrograms', fontsize=14)
plt.tight_layout()
plt.show()

## Part 4: SPD Manifold Features

The covariance matrix of mel-frequency bands over time is **symmetric positive definite** (SPD).
SPD matrices form a Riemannian manifold where the **log-Euclidean metric** is more
discriminative than the standard Frobenius norm. This captures correlations *between*
frequency bands — e.g., harmonics that move together — which flat spectrograms don't encode.

In [ ]:
class SPDBranch(nn.Module):
    """Extract and classify SPD covariance features from mel spectrograms.

    Pipeline:
    1. Group mel bins into n_bands frequency bands
    2. Compute covariance matrix across time [n_bands x n_bands]
    3. Apply log-Euclidean map: log(C) via eigendecomposition
    4. Extract upper triangle -> MLP -> embedding
    """

    def __init__(self, n_bands: int = 16, embed_dim: int = 32):
        super().__init__()
        self.n_bands = n_bands
        feat_dim = n_bands * (n_bands + 1) // 2
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, embed_dim),
        )

    def forward(self, mel_spec: torch.Tensor) -> torch.Tensor:
        """mel_spec: [B, 3, n_mels, T] -> embedding [B, embed_dim]."""
        spec = mel_spec[:, 0, :, :]  # [B, M, T]
        B, M, T = spec.shape

        # Group into bands
        bs = M // self.n_bands
        bands = spec[:, :self.n_bands * bs, :].reshape(B, self.n_bands, bs, T).mean(dim=2)

        # Covariance
        centered = bands - bands.mean(dim=-1, keepdim=True)
        cov = torch.bmm(centered, centered.transpose(1, 2)) / max(T - 1, 1)
        cov = cov + 1e-4 * torch.eye(self.n_bands, device=cov.device).unsqueeze(0)

        # Log-Euclidean map
        eigvals, eigvecs = torch.linalg.eigh(cov)
        log_cov = eigvecs @ torch.diag_embed(eigvals.clamp_min(1e-8).log()) @ eigvecs.transpose(-2, -1)

        # Upper triangle -> features
        idx = torch.triu_indices(self.n_bands, self.n_bands)
        features = log_cov[:, idx[0], idx[1]]

        return self.net(features)


# Test
spd = SPDBranch(n_bands=16, embed_dim=32)
dummy = torch.randn(4, 3, 128, 312)
out = spd(dummy)
print(f"SPD branch: input {dummy.shape} -> output {out.shape}")

## Part 5: Full Model

Architecture: `mel-spectrogram -> EfficientNet -> embedding -> Poincaré ball -> HyperbolicMLR`

The SPD branch runs in parallel and feeds into an ensemble combiner.
Both a hyperbolic and a standard Euclidean head are trained for ablation comparison.

In [ ]:
class GeometricBioacousticsModel(nn.Module):

    def __init__(self, num_classes: int, cfg: Config):
        super().__init__()
        self.cfg = cfg

        # Backbone
        self.backbone = timm.create_model(cfg.backbone, pretrained=True, num_classes=0)
        backbone_dim = self.backbone.num_features

        # Projector -> embedding space
        self.projector = nn.Sequential(
            nn.Linear(backbone_dim, cfg.embed_dim * 2),
            nn.BatchNorm1d(cfg.embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(cfg.embed_dim * 2, cfg.embed_dim),
        )

        # Hyperbolic head
        self.ball = PoincareBall(cfg.hyperbolic_c)
        self.hyp_head = HyperbolicMLR(cfg.embed_dim, num_classes, cfg.hyperbolic_c)

        # Euclidean head (for comparison)
        self.euc_head = nn.Linear(cfg.embed_dim, num_classes)

        # SPD branch
        self.spd_branch = SPDBranch(n_bands=cfg.spd_n_bands, embed_dim=32)

        # Ensemble: hyperbolic logits + SPD features -> final logits
        self.ensemble_head = nn.Linear(num_classes + 32, num_classes)

    def forward(self, mel_spec: torch.Tensor) -> Dict[str, torch.Tensor]:
        features = self.backbone(mel_spec)
        embedding = self.projector(features)

        # Hyperbolic path
        hyp_emb = self.ball.expmap0(embedding)
        hyp_logits = self.hyp_head(hyp_emb)

        # Euclidean path
        euc_logits = self.euc_head(embedding)

        # SPD path
        spd_feat = self.spd_branch(mel_spec)

        # Ensemble
        ens_logits = self.ensemble_head(torch.cat([hyp_logits, spd_feat], dim=-1))

        return {
            'hyp_logits': hyp_logits,
            'euc_logits': euc_logits,
            'ens_logits': ens_logits,
            'embedding': embedding,
            'hyp_embedding': hyp_emb,
        }


# Test
model = GeometricBioacousticsModel(num_classes, cfg).to(device)
dummy = torch.randn(2, 3, 128, 312).to(device)
out = model(dummy)
for k, v in out.items():
    print(f"  {k:15s}: {v.shape}")

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params:,}  Trainable: {trainable:,}")

## Part 6: Training

Multi-objective loss: 40% hyperbolic + 30% euclidean + 30% ensemble.
This lets us compare heads while training them jointly through the shared backbone.
Gradient clipping is essential for hyperbolic operations (gradients explode near the ball boundary).

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    all_labels, all_probs = [], []

    for mel, labels in loader:
        mel, labels = mel.to(device), labels.to(device)
        out = model(mel)

        loss = (
            0.4 * F.cross_entropy(out['hyp_logits'], labels) +
            0.3 * F.cross_entropy(out['euc_logits'], labels) +
            0.3 * F.cross_entropy(out['ens_logits'], labels)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        all_probs.append(F.softmax(out['ens_logits'], dim=-1).detach().cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    try:
        auc = roc_auc_score(
            np.eye(all_probs.shape[1])[all_labels], all_probs,
            average='macro', multi_class='ovr'
        )
    except Exception:
        auc = 0.0
    return total_loss / len(loader), auc


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_labels = []
    probs = {'hyp': [], 'euc': [], 'ens': []}
    all_emb = []

    for mel, labels in loader:
        mel, labels = mel.to(device), labels.to(device)
        out = model(mel)
        total_loss += F.cross_entropy(out['ens_logits'], labels).item()

        probs['hyp'].append(F.softmax(out['hyp_logits'], dim=-1).cpu().numpy())
        probs['euc'].append(F.softmax(out['euc_logits'], dim=-1).cpu().numpy())
        probs['ens'].append(F.softmax(out['ens_logits'], dim=-1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_emb.append(out['hyp_embedding'].cpu().numpy())

    all_labels = np.concatenate(all_labels)
    one_hot = np.eye(num_classes)[all_labels]
    results = {'loss': total_loss / len(loader), 'labels': all_labels,
               'embeddings': np.concatenate(all_emb)}

    for key in probs:
        p = np.concatenate(probs[key])
        try:
            results[f'auc_{key}'] = roc_auc_score(one_hot, p, average='macro', multi_class='ovr')
        except Exception:
            results[f'auc_{key}'] = 0.0
    return results

print("Training functions ready.")

In [ ]:
# ---- Create datasets ----
train_ds = SyntheticBirdDataset(species_list, samples_per_species=80, cfg=cfg, augment=True)
val_ds = SyntheticBirdDataset(species_list, samples_per_species=20, cfg=cfg, augment=False)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds)} samples ({len(val_loader)} batches)")

# ---- Create model ----
model = GeometricBioacousticsModel(num_classes, cfg).to(device)
model.hyp_head.init_from_taxonomy(tax_embeddings.to(device))

# ---- Optimizer: lower LR for pretrained backbone ----
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': cfg.lr * 0.1},
    {'params': model.projector.parameters()},
    {'params': model.hyp_head.parameters()},
    {'params': model.euc_head.parameters()},
    {'params': model.spd_branch.parameters()},
    {'params': model.ensemble_head.parameters()},
], lr=cfg.lr, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

# ---- Train ----
history = {
    'train_loss': [], 'train_auc': [], 'val_loss': [],
    'val_auc_hyp': [], 'val_auc_euc': [], 'val_auc_ens': []
}

print(f"\n{'Epoch':>5} | {'Train Loss':>10} {'Train AUC':>10} | {'Hyp AUC':>8} {'Euc AUC':>8} {'Ens AUC':>8}")
print('-' * 70)

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_auc = train_epoch(model, train_loader, optimizer, device)
    val = evaluate(model, val_loader, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val['loss'])
    history['val_auc_hyp'].append(val['auc_hyp'])
    history['val_auc_euc'].append(val['auc_euc'])
    history['val_auc_ens'].append(val['auc_ens'])

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:5d} | {train_loss:10.4f} {train_auc:10.3f} | "
              f"{val['auc_hyp']:8.3f} {val['auc_euc']:8.3f} {val['auc_ens']:8.3f}")

final = evaluate(model, val_loader, device)
print(f"\nFinal -> Hyperbolic: {final['auc_hyp']:.3f}  "
      f"Euclidean: {final['auc_euc']:.3f}  Ensemble: {final['auc_ens']:.3f}")

## Part 7: Visualization

In [ ]:
def plot_poincare_disk(embeddings, labels, species_list, title="Poincar\u00e9 Disk Embeddings"):
    """Visualize learned embeddings on the Poincare disk (first 2 dims)."""
    fig, ax = plt.subplots(figsize=(8, 8))
    theta = np.linspace(0, 2 * np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2, alpha=0.3)

    colors = plt.cm.Set1(np.linspace(0, 1, len(species_list)))
    for i, sp in enumerate(species_list):
        mask = labels == i
        if mask.sum() > 0:
            ax.scatter(
                embeddings[mask, 0], embeddings[mask, 1],
                c=[colors[i]], label=sp.replace('_', ' ').title(),
                s=30, alpha=0.6, edgecolors='none'
            )

    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax.set_title(title, fontsize=14)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()


def plot_training_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Val')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['val_auc_hyp'], 'g-', linewidth=2, label='Hyperbolic')
    axes[1].plot(epochs, history['val_auc_euc'], 'b--', linewidth=2, label='Euclidean')
    axes[1].plot(epochs, history['val_auc_ens'], 'r-', linewidth=2.5, label='Ensemble (Hyp+SPD)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Macro AUC')
    axes[1].set_title('Hyperbolic vs Euclidean vs Ensemble')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Plot!
plot_training_curves(history)
plot_poincare_disk(final['embeddings'], final['labels'], species_list)

## Part 8: TDA Analysis (Bonus)

Persistent homology on time-delay embeddings of the audio captures the **topology of the
vocal apparatus attractor**. Different species have different attractor shapes:
- Simple whistles = circle-like (one H1 feature)
- Trills = torus-like (multiple H1 features)
- Clicks = point-cloud-like (mostly H0)

This is too slow for per-batch training but useful for analysis and as a precomputed
ensemble feature on real data.

In [ ]:
from ripser import ripser


def time_delay_embedding(signal, delay=10, dim=3):
    """Takens' theorem: reconstruct the attractor from a 1D time series."""
    n = len(signal) - (dim - 1) * delay
    if n <= 0:
        return np.zeros((1, dim))
    embedded = np.zeros((n, dim))
    for d in range(dim):
        embedded[:, d] = signal[d * delay: d * delay + n]
    return embedded


def compute_tda_features(audio, sr, delay=10, dim=3, max_points=800):
    """Persistent homology features from audio time-delay embedding.

    Returns a 16-dim feature vector summarizing H0 and H1 topology.
    """
    cloud = time_delay_embedding(audio, delay=delay, dim=dim)
    if len(cloud) > max_points:
        idx = np.random.choice(len(cloud), max_points, replace=False)
        cloud = cloud[idx]
    cloud = (cloud - cloud.mean(0)) / (cloud.std(0) + 1e-10)

    result = ripser(cloud, maxdim=1, thresh=2.0)
    features = []
    for d in range(2):
        dgm = result['dgms'][d]
        finite = dgm[np.isfinite(dgm[:, 1])] if len(dgm) > 0 else np.empty((0, 2))
        if len(finite) == 0:
            features.extend([0.0] * 8)
            continue
        lifetimes = finite[:, 1] - finite[:, 0]
        features.extend([
            len(lifetimes),
            lifetimes.mean(),
            lifetimes.std() if len(lifetimes) > 1 else 0.0,
            lifetimes.max(),
            np.percentile(lifetimes, 75),
            finite[:, 0].mean(),
            (lifetimes ** 2).sum(),
            np.sqrt((lifetimes ** 2).sum()) / (len(lifetimes) + 1e-10),
        ])
    return np.array(features, dtype=np.float32)


# Compute TDA features for all species and visualize persistence diagrams
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
tda_all = {}

for i, (sp, ax) in enumerate(zip(species_list, axes.flat)):
    audio = generate_bird_call(sp, cfg.duration, cfg.sr)
    cloud = time_delay_embedding(audio, delay=10, dim=3)
    if len(cloud) > 800:
        idx = np.random.choice(len(cloud), 800, replace=False)
        cloud = cloud[idx]
    cloud = (cloud - cloud.mean(0)) / (cloud.std(0) + 1e-10)

    result = ripser(cloud, maxdim=1, thresh=2.0)
    feat = compute_tda_features(audio, cfg.sr)
    tda_all[sp] = feat

    # Plot persistence diagram
    for d, color in [(0, 'blue'), (1, 'red')]:
        dgm = result['dgms'][d]
        finite = dgm[np.isfinite(dgm[:, 1])]
        if len(finite) > 0:
            ax.scatter(finite[:, 0], finite[:, 1], c=color, s=10, alpha=0.5,
                       label=f'H{d}' if i == 0 else '')
    lim = ax.get_xlim()[1]
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3)
    ax.set_title(sp.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')

axes[0, 0].legend(fontsize=8)
plt.suptitle('Persistence Diagrams by Species\n(Blue=H0 connected components, Red=H1 loops)', fontsize=13)
plt.tight_layout()
plt.show()

# Show TDA feature vectors
print("TDA feature vectors (16-dim):")
print(f"{'Species':20s} | {'H0: count,mean,std,max':>30s} | {'H1: count,mean,std,max':>30s}")
print('-' * 85)
for sp in species_list:
    f = tda_all[sp]
    h0 = f"n={f[0]:.0f} mu={f[1]:.3f} sd={f[2]:.3f} mx={f[3]:.3f}"
    h1 = f"n={f[8]:.0f} mu={f[9]:.3f} sd={f[10]:.3f} mx={f[11]:.3f}"
    print(f"{sp:20s} | {h0:>30s} | {h1:>30s}")

## Part 9: Adapting for BirdCLEF+ 2025

To use this with real competition data:

```python
# 1. Load taxonomy (provides hierarchical structure for hyperbolic head)
import pandas as pd
tax_df = pd.read_csv('/kaggle/input/birdclef-2025/taxonomy.csv')
# Build dist_matrix from tax_df columns: order, family, genus, species

# 2. Replace SyntheticBirdDataset with:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, audio_dir, cfg):
        self.df = df
        self.audio_dir = Path(audio_dir)
        self.cfg = cfg
        self.label_map = {sp: i for i, sp in enumerate(df.primary_label.unique())}

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = librosa.load(
            self.audio_dir / row.filename,
            sr=self.cfg.sr, duration=self.cfg.duration
        )
        # ... same mel spectrogram pipeline as above ...
        return mel_tensor, self.label_map[row.primary_label]

# 3. Key competition tricks to add:
#    - Pre-train on BirdCLEF 2021-2024 historical data
#    - Pseudo-labeling on unlabeled soundscapes
#    - SoftAUCLoss instead of cross-entropy
#    - Ensemble 4-6 EfficientNet variants
#    - Silero VAD to filter human speech segments

# 4. Inference constraint: CPU-only, 90 minutes
#    - Hyperbolic head adds <1ms per sample (just distance computation)
#    - SPD branch adds ~2ms per sample (eigendecomposition)
#    - Both fit easily within the time budget
```

### Key hyperparameters to tune:
- `hyperbolic_c`: curvature (try 0.1, 0.5, 1.0, 2.0) — higher = more separation between orders
- `spd_n_bands`: number of frequency groups (8, 16, 32) — more = finer spectral covariance
- Loss weights between hyperbolic/euclidean/ensemble heads
- Whether to freeze backbone and only train geometric heads (transfer learning)

### References:
- Hyperbolic audio: [MERL, IEEE WASPAA 2023](https://ieeexplore.ieee.org/document/10248092/)
- Hyperbolic taxonomy: [NeurIPS 2025 BIOSCAN](https://arxiv.org/html/2508.16744v1)
- TDA for audio: [Spotify Research, SIAM J. Math. Data Sci.](https://epubs.siam.org/doi/10.1137/23M1605090)
- SPD manifolds for signals: [MDPI Entropy](https://pmc.ncbi.nlm.nih.gov/articles/PMC7597166/)